In [0]:
# SessionAnalysis
# 1.Calculate total time spent per session
# 2.Calculate total session time per user

from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import col, unix_timestamp, lag, expr, when
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, IntegerType
from datetime import datetime

spark = SparkSession.builder.appName("SessionAnalysis").getOrCreate()

data = [
    ("user1", "2023-08-26 10:10:00"),
    ("user1", "2023-08-26 10:10:25"),
    ("user2", "2023-08-26 12:00:00"),
    ("user2", "2023-08-26 12:10:00"),
    ("user1", "2023-08-26 14:30:00"),
    ("user1", "2023-08-26 16:00:00"),
    ("user2", "2023-08-26 16:30:00"),
    ("user1", "2023-08-26 18:00:00"),
]

schema = StructType([
    StructField("userid", StringType(), nullable=False),
    StructField("timestamp", TimestampType(), nullable=False),
])

rows = [(row[0], datetime.strptime(row[1], "%Y-%m-%d %H:%M:%S")) for row in data]
df = spark.createDataFrame(rows, schema=schema)
df.show()

window_spec = Window().partitionBy("userid").orderBy("timestamp")
df = df.withColumn("session_gap", unix_timestamp("timestamp") - unix_timestamp(lag("timestamp").over(window_spec)))
df = df.withColumn("sessionid", when(col("session_gap") > 30, None).otherwise(col("userid")) + expr("'_session'") + expr("cast(row_number() over (partition by userid order by timestamp) as string)"))
df.show()

# 1.Calculate total time spent per session
session_window = Window().partitionBy("userid", "sessionid").orderBy("timestamp")
df = df.withColumn("session_start", when(col("session_gap") > 30, None).otherwise(lag("timestamp").over(session_window)))
df = df.withColumn("session_duration", when(col("session_start").isNotNull(), (col("timestamp").cast("long") - col("session_start").cast("long"))).otherwise(0))
df.show()

# 2.Calculate total session time per user
user_window = Window().partitionBy("userid")
df = df.withColumn("total_session_time", expr("sum(session_duration) over (partition by userid)"))
df.show()


+------+-------------------+
|userid|          timestamp|
+------+-------------------+
| user1|2023-08-26 10:10:00|
| user1|2023-08-26 10:10:25|
| user2|2023-08-26 12:00:00|
| user2|2023-08-26 12:10:00|
| user1|2023-08-26 14:30:00|
| user1|2023-08-26 16:00:00|
| user2|2023-08-26 16:30:00|
| user1|2023-08-26 18:00:00|
+------+-------------------+

+------+-------------------+-----------+---------+
|userid|          timestamp|session_gap|sessionid|
+------+-------------------+-----------+---------+
| user1|2023-08-26 10:10:00|       null|     null|
| user1|2023-08-26 10:10:25|         25|     null|
| user1|2023-08-26 14:30:00|      15575|     null|
| user1|2023-08-26 16:00:00|       5400|     null|
| user1|2023-08-26 18:00:00|       7200|     null|
| user2|2023-08-26 12:00:00|       null|     null|
| user2|2023-08-26 12:10:00|        600|     null|
| user2|2023-08-26 16:30:00|      15600|     null|
+------+-------------------+-----------+---------+

+------+-------------------+----------